# ZettelForge: CTI Analysis Workflow

A complete cyber threat intelligence analysis pipeline using ZettelForge's agentic memory system.

What this notebook covers:

1. **Install and import** ZettelForge
2. **Ingest** three+ realistic CTI reports (APT28, Lazarus, Volt Typhoon)
3. **Extracted entities** shown for each ingested report
4. **Search by threat actor** using `recall_actor()`
5. **Semantic recall** using `recall()`
6. **Knowledge graph statistics** via `get_stats()`
7. **Synthesize findings** using `synthesize()`
8. **Clean up** temporary storage

This notebook uses **mock LLM and embedding providers** so it runs portably without any API keys or GPU. All data lives in a temporary directory and is cleaned up at the end.

---

## 1. Install and import

Install ZettelForge if it is not already installed, then configure mock providers and import `MemoryManager`.

In [ ]:
# Install ZettelForge (skip if already installed)
import subprocess, sys
try:
    import zettelforge
    print(f"ZettelForge {zettelforge.__version__} already installed")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "zettelforge"])
    import zettelforge
    print(f"Installed ZettelForge {zettelforge.__version__}")

In [ ]:
import os
import tempfile

# Use mock providers so no API keys or GPU are needed
os.environ["ZETTELFORGE_LLM_PROVIDER"] = "mock"
os.environ["ZETTELFORGE_EMBEDDING_PROVIDER"] = "mock"

# All data goes to a temp directory -- no persistent state
TMPDIR = tempfile.mkdtemp(prefix="zettelforge_cti_")
os.environ["AMEM_DATA_DIR"] = TMPDIR

from zettelforge import MemoryManager

mm = MemoryManager(jsonl_path=f"{TMPDIR}/notes.jsonl")
print(f"MemoryManager ready. Data dir: {TMPDIR}")
print(f"LLM provider: mock   |   Embedding provider: mock")

---

## 2. Ingest CTI reports

We ingest three realistic threat intelligence reports covering distinct actors and vulnerability classes. Each call to `remember()` returns a `MemoryNote` with auto-extracted entities.

In [ ]:
# Report 1: APT28 (Fancy Bear / STRONTIUM) -- Russian state-sponsored actor
report_apt28 = (
    "APT28 (also known as Fancy Bear, STRONTIUM, Sofacy) is a Russian state-sponsored "
    "threat actor attributed to the Russian General Staff Main Intelligence Directorate "
    "(GRU). The group has been observed using Cobalt Strike for lateral movement in "
    "targeted campaigns against NATO-affiliated organizations. They exploit "
    "CVE-2024-3094 (XZ Utils backdoor) for initial access, then deploy XAgent malware "
    "for persistence and data exfiltration. MITRE ATT&CK technique T1021 (Remote Services) "
    "is their primary lateral movement method. Spear-phishing campaigns use credential-harvesting "
    "links and OAuth application abuse to compromise defense contractor environments."
)

note1, status1 = mm.remember(report_apt28, domain="cti", source_type="report", source_ref="cti_analysis.ipynb/report_1")

print(f"Status: {status1}  |  Note ID: {note1.id}")
print(f"Entities: {note1.semantic.entities}")

In [ ]:
# Report 2: Lazarus Group -- DPRK-aligned threat actor targeting cryptocurrency
report_lazarus = (
    "Lazarus Group (DPRK-aligned, also tracked as HIDDEN COBRA) conducted a supply chain "
    "attack targeting cryptocurrency exchanges in Q1 2025. The campaign used trojanized "
    "npm packages to deliver FALLCHILL malware. Indicators of compromise include the IP "
    "address 185.29.8.18 and the domain update-check.github-cloud[.]com. CVE-2023-42793 "
    "was exploited in the JetBrains TeamCity component to gain initial access. "
    "MITRE ATT&CK techniques T1195.002 (Supply Chain Compromise) and T1566.001 "
    "(Spearphishing Attachment) were observed. A novel macOS payload was delivered via "
    "fake recruiter LinkedIn outreach using signed but malicious .pkg installers."
)

note2, status2 = mm.remember(report_lazarus, domain="cti", source_type="report", source_ref="cti_analysis.ipynb/report_2")

print(f"Status: {status2}  |  Note ID: {note2.id}")
print(f"Entities: {note2.semantic.entities}")

In [ ]:
# Report 3: Volt Typhoon -- PRC-aligned actor targeting US critical infrastructure
report_volt = (
    "Volt Typhoon (also tracked as UNC3236) is a PRC-aligned threat actor that continues "
    "to target US critical infrastructure, focusing on communications, energy, and water "
    "sectors. The group uses living-off-the-land techniques, leveraging built-in Windows "
    "tools like netsh and PowerShell for tunneling and defense evasion. No custom malware "
    "is deployed, making traditional signature-based detection difficult. MITRE ATT&CK "
    "techniques T1059.001 (PowerShell) and T1018 (Remote System Discovery) are "
    "characteristic of their operations. Pre-positioning activity has been observed on "
    "critical infrastructure networks since at least 2021."
)

note3, status3 = mm.remember(report_volt, domain="cti", source_type="report", source_ref="cti_analysis.ipynb/report_3")

print(f"Status: {status3}  |  Note ID: {note3.id}")
print(f"Entities: {note3.semantic.entities}")

---

## 3. Search and recall by threat actor

ZettelForge indexes entities during ingest. You can query by threat actor name, CVE ID, or MITRE ATT&CK technique ID for fast lookups.

In [ ]:
# Recall by threat actor -- searches across actor / threat_actor / intrusion_set entity types
results = mm.recall_actor("apt28", k=5)
print(f"recall_actor('apt28') returned {len(results)} note(s)\n")
for i, n in enumerate(results, 1):
    print(f"  [{i}] {n.content.raw[:120]}...")

In [ ]:
# Recall by CVE ID
cve_results = mm.recall_cve("CVE-2024-3094", k=5)
print(f"recall_cve('CVE-2024-3094') returned {len(cve_results)} note(s)\n")
for i, n in enumerate(cve_results, 1):
    print(f"  [{i}] {n.content.raw[:120]}...")

---

## 4. Semantic recall

Beyond exact entity lookup, `recall()` performs blended vector + graph retrieval. This finds conceptually related content even if no exact entity match exists.

In [ ]:
semantic_results = mm.recall("supply chain attacks on open source software", k=5)
print(f"recall('supply chain attacks on open source software') returned {len(semantic_results)} note(s)\n")
for i, n in enumerate(semantic_results, 1):
    print(f"  [{i}] {n.content.raw[:120]}...")

In [ ]:
# Another semantic query
results2 = mm.recall("critical infrastructure targeting", k=5)
print(f"recall('critical infrastructure targeting') returned {len(results2)} note(s)\n")
for i, n in enumerate(results2, 1):
    print(f"  [{i}] {n.content.raw[:120]}...")

---

## 5. Knowledge graph statistics

ZettelForge builds a knowledge graph as notes are ingested. Entity-to-entity edges (actor USES_TOOL, actor EXPLOITS_CVE) are inferred heuristically. Call `get_stats()` to inspect the graph state.

In [ ]:
stats = mm.get_stats()

print(f"Total notes: {stats['total_notes']}")
print(f"Retrievals served: {stats.get('retrievals', 0)}")
print(f"Entity index hits: {stats.get('entity_index_hits', 0)}")
print()

entity_index = stats.get("entity_index", {})
if entity_index:
    total_entities = sum(v.get("unique_entities", 0) for v in entity_index.values())
    total_mappings = sum(v.get("total_mappings", 0) for v in entity_index.values())
    print(f"Entity index: {total_entities} unique entities across {total_mappings} note mappings")
    for etype, info in sorted(entity_index.items()):
        if info.get("unique_entities", 0) > 0:
            print(f"    {etype}: {info['unique_entities']} entities, {info['total_mappings']} mappings")

---

## 6. Synthesize findings

The `synthesize()` method retrieves relevant notes and runs LLM-based synthesis to produce a structured answer. With the mock LLM provider it returns a canned response; in production you would wire a real LLM (OpenAI, Anthropic, Ollama, etc.) via config.

In [ ]:
# Synthesize a direct answer from stored intelligence
# Wrapped in try/except since the mock LLM returns a simple string, not JSON
try:
    synthesis = mm.synthesize("What do we know about APT28 spear-phishing tradecraft?", k=5)
    print(f"Query: {synthesis.get('query', 'N/A')}")
    print(f"Format: {synthesis.get('format', 'N/A')}")
    raw = synthesis.get("synthesis", synthesis)
    if isinstance(raw, dict):
        print(f"Answer: {raw.get('answer', raw.get('summary', str(raw)[:300]))}")
    else:
        print(f"Synthesis: {str(raw)[:300]}")
    print(f"Sources count: {synthesis.get('metadata', {}).get('sources_count', 0)}")
except Exception as e:
    print(f"Synthesis encountered an expected issue with mock provider: {e}")
    print("In production, configure ZETTELFORGE_LLM_PROVIDER=openai (or ollama, anthropic, etc.)")

In [ ]:
# A broader synthesis across all stored intelligence
try:
    synthesis2 = mm.synthesize("Summarize the supply chain attack patterns across all actors", k=10)
    raw2 = synthesis2.get("synthesis", synthesis2)
    if isinstance(raw2, dict):
        print(f"Synthesis answer: {raw2.get('answer', raw2.get('summary', str(raw2)[:400]))}")
    else:
        print(f"Synthesis: {str(raw2)[:400]}")
except Exception as e:
    print(f"Synthesis encountered an expected issue with mock provider: {e}")

---

## 7. Clean up

Remove the temporary directory since all state is ephemeral.

In [ ]:
import shutil
shutil.rmtree(TMPDIR, ignore_errors=True)
print(f"Cleaned up temporary data directory: {TMPDIR}")

---

## Summary

This notebook demonstrated the core ZettelForge CTI analysis workflow:

- **`MemoryManager.remember()`** ingests unstructured CTI reports and auto-extracts entities (threat actors, CVEs, tools, techniques).
- **`recall_actor()`**, **`recall_cve()`**, and **`recall_technique()`** provide fast entity-based lookup.
- **`recall()`** performs blended semantic + graph retrieval for open-ended queries.
- **`get_stats()`** inspects the knowledge graph built automatically during ingest.
- **`synthesize()`** runs LLM-backed synthesis over retrieved notes for analyst-ready answers.

To use with real LLMs and embeddings, set:

```bash
export ZETTELFORGE_LLM_PROVIDER=openai
export ZETTELFORGE_EMBEDDING_PROVIDER=fastembed
```

or configure via the `zettelforge.config` API.